# Vectorización de Texto: Bag of Words y TF-IDF

---

**Objetivo:** Comparar las técnicas de vectorización por conteo (BoW) y por ponderación (TF-IDF), entendiendo cuándo es conveniente usar cada una y cómo capturar el contexto con N-gramas.

**Resultados de aprendizaje:** Al finalizar este cuaderno, vas a poder:
1. Diferenciar entre la frecuencia simple de términos y su importancia relativa en un corpus.
2. Implementar y comparar BoW y TF-IDF usando `scikit-learn`.
3. Identificar la utilidad de los N-gramas para capturar términos compuestos.
4. Interpretar matrices esparsas y visualizarlas para el análisis de texto.

**Relación con cuadernos anteriores:** Ya vimos cómo funciona la "Bolsa de Palabras". En este encuentro vamos a sofisticar nuestra técnica para que el algoritmo entienda qué palabras son realmente importantes y cuáles son solo "ruido" frecuente.

In [ ]:
# --- 0. PREPARACIÓN DEL ENTORNO ---
# Importamos las librerías necesarias para el análisis.

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- DEFINICIÓN DEL CORPUS ---
# Usamos un corpus pequeño y controlado para que los resultados sean fáciles de leer.
corpus_estudio = [
    "El perro muerde al hombre",
    "El hombre muerde al perro",
    "El perro come carne",
    "El hombre come comida"
]

print("Nuestro corpus de estudio:")
for i, documento in enumerate(corpus_estudio):
    print(f"Documento {i+1}: {documento}")

## Sección 1: Bag of Words (BoW) - Repaso

Como recordás, la técnica Bag of Words cuenta cuántas veces aparece cada palabra en un documento.

**Características:**
*   **Ignora el orden:** Lo que importa es la presencia y frecuencia.
*   **Crea un vocabulario global:** Mapea todas las palabras únicas del corpus.
*   **Dificultad:** Da el mismo peso a palabras muy comunes (como "el" o "al") que a palabras con mucho significado (como "muerde" o "carne").

In [ ]:
# --- 1.1. Implementación de Bag of Words ---
print("\n--- PROCESANDO CON BAG OF WORDS ---")

# 1. Inicializamos y entrenamos el contador
contador_bow = CountVectorizer()
matriz_bow = contador_bow.fit_transform(corpus_estudio)

# 2. Obtenemos el vocabulario identificado
vocabulario = contador_bow.get_feature_names_out()

# 3. Visualizamos la matriz de conteos en una tabla
tabla_bow = pd.DataFrame(
    matriz_bow.toarray(), 
    columns=vocabulario, 
    index=[f"Doc {i+1}" for i in range(len(corpus_estudio))]
)

print("Matriz de frecuencias simple (BoW):")
print(tabla_bow)

## Sección 2: TF-IDF - Ponderando la Importancia

El problema de BoW es que las palabras que aparecen en todos lados (Stopwords) dominan los conteos. **TF-IDF** (Term Frequency-Inverse Document Frequency) es una mejora inteligente.

### ¿Cómo funciona?
*   **Term Frequency (TF):** Mirá cuántas veces aparece una palabra en un documento (igual que BoW).
*   **Inverse Document Frequency (IDF):** Mirá qué tan "rara" es esa palabra en todo el corpus. Si una palabra aparece en todos los documentos (como "el"), su importancia baja. Si aparece en muy pocos, su importancia sube.

La fórmula es: **Frecuencia * Importancia**. Así, las palabras que mejor definen a cada documento tienen los pesos más altos.

In [ ]:
# --- 2.1. Implementación de TF-IDF ---
print("\n--- PROCESANDO CON TF-IDF ---")

# 1. Inicializamos el vectorizador TF-IDF
contador_tfidf = TfidfVectorizer()

# 2. Aprendemos y transformamos el corpus
matriz_tfidf = contador_tfidf.fit_transform(corpus_estudio)
vocabulario_tfidf = contador_tfidf.get_feature_names_out()

# 3. Visualizamos los pesos resultantes
tabla_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(), 
    columns=vocabulario_tfidf, 
    index=[f"Doc {i+1}" for i in range(len(corpus_estudio))]
)

print("Matriz de pesos TF-IDF (Fijate cómo bajan los valores de 'el'):")
print(tabla_tfidf.round(3))

## Sección 3: Comparativa Visual y Análisis

Mirá la diferencia entre los conteos simples y los pesos ponderados para entender por qué TF-IDF es la técnica preferida en muchos casos de PLN.

In [ ]:
# --- 3.1. Visualización Comparativa ---
doc_a_comparar = 2 # Analizamos el Documento 3 ("El perro come carne")

# Preparamos los datos para el gráfico
analisis_bow = pd.DataFrame({'palabra': vocabulario, 'valor': matriz_bow[doc_a_comparar].toarray()[0]})
analisis_tfidf = pd.DataFrame({'palabra': vocabulario_tfidf, 'valor': matriz_tfidf[doc_a_comparar].toarray()[0]})

# Creamos la figura
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
sns.barplot(x='palabra', y='valor', data=analisis_bow.sort_values('valor', ascending=False))
plt.title(f'Bag of Words (Conteos) - Doc {doc_a_comparar+1}')
plt.xticks(rotation=45, ha='right')
plt.ylabel("Cantidad")

plt.subplot(1, 2, 2)
sns.barplot(x='palabra', y='valor', data=analisis_tfidf.sort_values('valor', ascending=False))
plt.title(f'TF-IDF (Pesos) - Doc {doc_a_comparar+1}')
plt.xticks(rotation=45, ha='right')
plt.ylabel("Importancia Relativa")

plt.suptitle(f"Análisis del Documento {doc_a_comparar+1}: '{corpus_estudio[doc_a_comparar]}'", fontsize=16, fontweight="bold")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("Observaciones:")
print("- En BoW, todas las palabras parecen igual de importantes (todas valen 1).")
print("- En TF-IDF, 'carne' y 'come' resaltan más que 'el' o 'perro'.")
print("  Esto sucede porque 'el' y 'perro' se repiten en muchos documentos del corpus.")

## Sección 4: Yendo más allá - N-gramas

Hasta ahora, venimos tratando cada palabra de forma individual (Unigramas). Sin embargo, perdemos el significado de términos compuestos como "Buenos Aires" o "Mar del Plata".

Los **N-gramas** son secuencias contiguas de palabras:
*   **Bigramas (N=2):** Pares de palabras ("Buenos Aires").
*   **Trigramas (N=3):** Tríos de palabras ("Mar del Plata").

Mirá cómo cambia nuestro vocabulario cuando le pedimos a Python que capture estas secuencias.

In [ ]:
# --- 4.1. Preparación de datos con términos compuestos ---
corpus_ciudades = [
    "Buenos Aires es la capital de Argentina",
    "Visité Mar del Plata y Buenos Aires el verano pasado"
]

print("Nuevo corpus con nombres compuestos:")
for d in corpus_ciudades: print(f"- {d}")

In [ ]:
# --- 4.2. El problema de los Unigramas ---
print("\n1. Analizando solo con Unigramas (Palabras sueltas):")

contador_simple = CountVectorizer()
matriz_simple = contador_simple.fit_transform(corpus_ciudades)
vocab_simple = contador_simple.get_feature_names_out()

# Filtramos términos de interés para comparar
terminos_interes = ['buenos', 'aires', 'mar', 'del', 'plata']
vocab_filtrado = [p for p in vocab_simple if p in terminos_interes]

print(f"Vocabulario identificado: {vocab_filtrado}")
print("-> Notá que 'buenos' y 'aires' están separados. Perdimos el concepto de la ciudad.")

In [ ]:
# --- 4.3. La solución con N-gramas (Unigramas + Bigramas + Trigramas) ---
print("\n2. Analizando con N-gramas (unigramas, bigramas y trigramas):")

# Configuramos ngram_range=(1, 3) para capturar secuencias de 1, 2 y 3 palabras
contador_ngramas = CountVectorizer(ngram_range=(1, 3))
matriz_ngramas = contador_ngramas.fit_transform(corpus_ciudades)
vocab_ngramas = contador_ngramas.get_feature_names_out()

print("Vocabulario con N-gramas (algunos ejemplos):")
frases_clave = ['buenos', 'aires', 'buenos aires', 'mar del plata']
vocab_filtrado_ngram = [p for p in vocab_ngramas if p in frases_clave]
print(vocab_filtrado_ngram)

print("\nMatriz de conteos (extracto para el Doc 2):")
tabla_ngramas = pd.DataFrame(matriz_ngramas.toarray(), columns=vocab_ngramas, index=["Doc 1", "Doc 2"])
columnas_mostrar = [c for c in ['buenos aires', 'mar del plata', 'verano pasado'] if c in vocab_ngramas]
print(tabla_ngramas.loc[["Doc 2"], columnas_mostrar])

## Conclusiones Clave

*   **BoW (Bolsa de Palabras):** Es rápido y simple, pero no distingue importancia entre palabras.
*   **TF-IDF:** Es más sofisticado y destaca los términos más relevantes de cada documento dentro de un corpus.
*   **N-gramas:** Permiten capturar el contexto de términos compuestos, evitando que conceptos como "Buenos Aires" se rompan en partes sin sentido.

### Glosario Técnico
*   **Token:** Unidad mínima de procesamiento (generalmente una palabra).
*   **Corpus:** Conjunto de documentos a analizar.
*   **Matriz Esparsa (Sparse):** Formato eficiente de almacenamiento donde solo se guardan los valores que NO son cero.
*   **Stopwords:** Palabras muy frecuentes (artículos, preposiciones) que suelen filtrarse.

---
**Actividad:** Intentá cambiar el `ngram_range` en el código de arriba y observá cómo explota la cantidad de términos en el vocabulario. ¿Por qué creés que no siempre es bueno usar N-gramas muy largos?